# 🎯 Prompt Tuning with PEFT (Parameter-Efficient Fine-Tuning)

**Prompt Tuning** is a lightweight fine-tuning technique where we prepend a set of learnable "virtual tokens" (soft prompts) to the input, while keeping the entire pre-trained language model **frozen**. Only these virtual token embeddings are updated during training, making it extremely parameter-efficient.

### What we'll cover in this notebook:
1. **Setup** — Import all required libraries
2. **Dataset Preparation** — Load and preprocess the Twitter Complaints dataset
3. **Model & PEFT Configuration** — Configure Prompt Tuning and wrap the base model
4. **Before Tuning** — Query the model on multiple test samples to see raw (untrained) responses
5. **Training Loop** — Train the soft prompt embeddings
6. **After Tuning** — Query the same samples again to compare the improvement
7. **Save & Reload** — Persist the tiny PEFT checkpoint and reload for inference

| Detail | Value |
|---|---|
| **Base Model** | `Mistral-7B-v0.1` |
| **Task** | Binary text classification — *complaint* vs *no complaint* |
| **Dataset** | `twitter_complaints` from the RAFT benchmark |
| **Technique** | Prompt Tuning (PEFT) |

---
## 1. Setup & Imports

We import the following key libraries:
- **`transformers`** — Hugging Face library for loading pre-trained models and tokenizers
- **`peft`** — Parameter-Efficient Fine-Tuning library that provides Prompt Tuning, LoRA, etc.
- **`datasets`** — For loading benchmark datasets from the Hugging Face Hub
- **`torch`** — PyTorch, the deep learning framework
- **`tqdm`** — Progress bars for training loops

In [ ]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType, PeftType
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import default_data_collator, get_linear_schedule_with_warmup
from tqdm import tqdm

---
## 2. Configuration & Hyperparameters

We define all key settings for the experiment in one place for easy modification:

| Parameter | Value | Purpose |
|---|---|---|
| `model_name_or_path` | Mistral-7B-v0.1 | Base causal LM (frozen during training) |
| `dataset_name` | twitter_complaints | Binary classification task from RAFT |
| `max_length` | 64 | Max token length for input sequences |
| `lr` | 1e-4 | Learning rate for the soft prompt embeddings |
| `num_epochs` | 10 | Number of training epochs |
| `batch_size` | 8 | Batch size for training and evaluation |

In [ ]:
seed = 42
device = "cuda"
model_name_or_path = "mistralai/Mistral-7B-v0.1"
tokenizer_name_or_path = "mistralai/Mistral-7B-v0.1"
dataset_name = "twitter_complaints"
text_column = "Tweet text"
label_column = "text_label"
max_length = 64
lr = 1e-4
num_epochs = 10
batch_size = 8
set_seed(seed)

---
## 3. Dataset Preparation

### 3.1 Load the Dataset

We use the **`twitter_complaints`** subset from the [RAFT benchmark](https://huggingface.co/datasets/ought/raft). Each tweet is labeled as either:
- `complaint` — The tweet contains a complaint
- `no complaint` — The tweet does not contain a complaint

We also map the numeric `Label` column to human-readable text labels (`text_label`).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ought/raft", dataset_name)

# Convert numeric labels to human-readable text labels
classes = [k.replace("_", " ") for k in dataset["train"].features["Label"].names]
print("Classes:", classes)

dataset = dataset.map(
    lambda x: {"text_label": [classes[label] for label in x["Label"]]},
    batched=True,
    num_proc=1,
)
print(dataset)
dataset["train"][0]

#### Check label distribution

Let's verify the class balance in our training set. RAFT tasks typically have very few training examples (50 per task), which makes parameter-efficient methods like Prompt Tuning especially appealing.

In [ ]:
from collections import Counter
Counter(dataset["train"]["Label"])

### 3.2 Preprocess the Dataset

For causal LM-based classification, we format each sample as:

```
Tweet text : <tweet content>
Label : <complaint / no complaint>
```

Key preprocessing steps:
1. **Tokenize** the input prompt and the target label separately
2. **Concatenate** them so the model learns to *generate* the label after seeing the prompt
3. **Mask the prompt tokens** in the labels with `-100` so the loss is only computed on the label tokens
4. **Left-pad** all sequences to `max_length` (left-padding is standard for causal / decoder-only models)

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

target_max_length = max([len(tokenizer(class_label)["input_ids"]) for class_label in classes])
print(f"{target_max_length=}")


def preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [f"{text_column} : {x}\nLabel : " for x in examples[text_column]]
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(inputs)
    labels = tokenizer(targets, add_special_tokens=False)  # don't add bos token because we concatenate with inputs
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.eos_token_id]
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_length - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = torch.tensor(model_inputs["input_ids"][i][:max_length])
        model_inputs["attention_mask"][i] = torch.tensor(model_inputs["attention_mask"][i][:max_length])
        labels["input_ids"][i] = torch.tensor(labels["input_ids"][i][:max_length])
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


processed_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

train_dataset = processed_datasets["train"]
eval_dataset = processed_datasets["train"]


train_dataloader = DataLoader(
    train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)
next(iter(train_dataloader))

### 3.3 Preprocess the Test Dataset

For the test set, we only tokenize the **input prompt** (without the label). The model will generate the label at inference time. Note that we use the same left-padding strategy.

In [ ]:
def test_preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [f"{text_column} : {x}\nLabel : " for x in examples[text_column]]
    model_inputs = tokenizer(inputs)
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        model_inputs["input_ids"][i] = torch.tensor(model_inputs["input_ids"][i][:max_length])
        model_inputs["attention_mask"][i] = torch.tensor(model_inputs["attention_mask"][i][:max_length])
    return model_inputs


test_dataset = dataset["test"].map(
    test_preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

test_dataloader = DataLoader(test_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)
next(iter(test_dataloader))

---
## 4. Prompt Tuning Configuration & Model Setup

### 4.1 Define the Prompt Tuning Config

Key parameters:
- **`task_type=CAUSAL_LM`** — We're using a causal (autoregressive) language model
- **`prompt_tuning_init=TEXT`** — Initialize the soft prompt from a meaningful text string (rather than random). This gives training a head start.
- **`num_virtual_tokens`** — Number of learnable tokens prepended to the input. We set this equal to the token count of our init text.
- **`prompt_tuning_init_text`** — The text used to initialize the virtual token embeddings

In [ ]:
prompt_tuning_init_text = "Classify if the tweet is a complaint or no complaint.\n"

peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=len(tokenizer(prompt_tuning_init_text)["input_ids"]),
    prompt_tuning_init_text=prompt_tuning_init_text,
    tokenizer_name_or_path=model_name_or_path,
)

print(f"Number of virtual tokens: {peft_config.num_virtual_tokens}")

### 4.2 Hugging Face Hub Login

Required to download gated models like Mistral-7B and to push checkpoints to the Hub later.

In [ ]:
from huggingface_hub import login
login()

### 4.3 Load the Base Model & Wrap with PEFT

We load Mistral-7B in `float16` precision, then wrap it with `get_peft_model()`. This:
1. **Freezes all original model parameters** (no gradients computed for them)
2. **Adds learnable soft prompt embeddings** as the only trainable parameters

Notice how the trainable parameters are a tiny fraction of the total — that's the power of Prompt Tuning!

In [ ]:
# Load the base causal LM
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype=torch.float16)

# Wrap with PEFT — this freezes the base model and adds trainable soft prompt
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Enable gradient checkpointing to save GPU memory during training
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = model.to(device)

#### Inspect the model architecture

Notice the `PromptEmbedding` layer added at the top — these are our learnable virtual tokens.

In [ ]:
model

### 4.4 Setup Optimizer & Learning Rate Scheduler

We use **AdamW** optimizer with weight decay. Even though only the soft prompt is trainable, we still pass `model.parameters()` — PyTorch will only update parameters that have `requires_grad=True`.

The **linear schedule with warmup** gradually decays the learning rate from `lr` to 0 over the training steps.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

---
## 5. 🔍 BEFORE Tuning — Query the Model on Test Samples

Let's see how the model responds **before any fine-tuning**. Since the soft prompt embeddings are just initialized (not trained yet), the model essentially behaves like the raw base model.

We'll run inference on **multiple test samples** so we can later compare with post-tuning results.

In [ ]:
# Define the test sample indices we want to compare before and after tuning
test_indices = [4, 10, 18, 25, 33, 40]

model.eval()
print("=" * 80)
print("MODEL RESPONSES **BEFORE** PROMPT TUNING")
print("=" * 80)

before_results = {}

for i in test_indices:
    tweet = dataset["test"][i]["Tweet text"]
    input_text = f'{text_column} : {tweet}\nLabel : '
    inputs = tokenizer(input_text, return_tensors="pt")

    with torch.no_grad():
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=20,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    # Extract only the generated part (after the prompt)
    response = generated[len(input_text):].strip()
    before_results[i] = response

    print(f"\n--- Test Sample {i} ---")
    print(f"Tweet   : {tweet[:100]}{'...' if len(tweet) > 100 else ''}")
    print(f"Response: {response}")

print("\n" + "=" * 80)
print("Observation: The model generates arbitrary text instead of 'complaint' / 'no complaint'.")
print("=" * 80)

---
## 6. 🏋️ Training Loop

Now we train **only the soft prompt embeddings** for `num_epochs` epochs. The entire base model remains frozen.

For each epoch we:
1. **Train** — Forward pass → compute loss (only on label tokens) → backward pass → update soft prompt
2. **Evaluate** — Compute validation loss and perplexity on the eval set

We track **training loss**, **eval loss**, and **perplexity** to monitor convergence.

In [ ]:
# Training and evaluation loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Training")):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(dtype=torch.float16, device_type="cuda"):
            outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Evaluating")):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

---
## 7. 🔍 AFTER Tuning — Query the Same Test Samples

Now let's run inference on the **exact same test samples** we queried before training. This lets us directly compare the model's behavior before and after Prompt Tuning.

We expect the model to now generate clean labels: `complaint` or `no complaint`.

In [ ]:
model.eval()
print("=" * 80)
print("MODEL RESPONSES **AFTER** PROMPT TUNING")
print("=" * 80)

after_results = {}

for i in test_indices:
    tweet = dataset["test"][i]["Tweet text"]
    input_text = f'{text_column} : {tweet}\nLabel : '
    inputs = tokenizer(input_text, return_tensors="pt")

    with torch.no_grad():
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=5,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    response = generated[len(input_text):].strip()
    after_results[i] = response

    print(f"\n--- Test Sample {i} ---")
    print(f"Tweet   : {tweet[:100]}{'...' if len(tweet) > 100 else ''}")
    print(f"Response: {response}")

print("\n" + "=" * 80)
print("The model now generates proper classification labels!")
print("=" * 80)

### 7.1 Side-by-Side Comparison: Before vs After

Let's print a clean comparison table to see the improvement at a glance.

In [ ]:
print(f"{'Sample':>6} | {'BEFORE Tuning':<40} | {'AFTER Tuning':<20}")
print("-" * 75)
for i in test_indices:
    before = before_results[i][:38]
    after = after_results[i][:18]
    print(f"{i:>6} | {before:<40} | {after:<20}")

---
## 8. 💾 Save the Model

Since only the soft prompt embeddings are trainable, the saved checkpoint is **extremely small** (a few KB) compared to the full model (14 GB+). You can either:

- **Push to Hugging Face Hub** — for sharing and versioning
- **Save locally** — for quick local reuse

**Option 1:** Push to Hugging Face Hub
```python
model.push_to_hub("mistral_prompt_tuning", private=True)
```

**Option 2:** Save locally
```python
peft_model_id = "mistral_prompt_tuning"
model.save_pretrained(peft_model_id)
```

In [ ]:
# Save / push the model (uncomment the option you prefer)

# Option 1: Push to Hub
# model.push_to_hub("mistral_prompt_tuning", private=True)

# Option 2: Save locally
peft_model_id = "mistral_prompt_tuning"
model.save_pretrained(peft_model_id)

### Check the size of the checkpoint

The PEFT checkpoint only contains the soft prompt embeddings — typically just a few kilobytes!

In [ ]:
!ls -lh mistral_prompt_tuning/

In [ ]:
!nvidia-smi

---
## 9. 🔄 Reload the PEFT Checkpoint & Inference

To demonstrate that our tiny checkpoint works independently, we reload everything from scratch:
1. Load the PEFT config to find which base model was used
2. Load the base model
3. Load the PEFT adapter (soft prompt) on top
4. Run inference to verify it still works

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

dataset = load_dataset("ought/raft", "twitter_complaints")
peft_model_id = "mistral_prompt_tuning"  # local path or Hub repo
device = "cuda"
text_column = "Tweet text"
label_column = "text_label"

# Load config, base model, and PEFT adapter
config = PeftConfig.from_pretrained(peft_model_id)
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(model, peft_model_id)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

### Run inference with the reloaded model

Let's verify the reloaded model produces the same correct classifications.

In [ ]:
model.to(device)
model.eval()

print("Inference with reloaded PEFT checkpoint:")
print("=" * 60)

for i in [4, 10, 18, 25, 33, 40]:
    tweet = dataset["test"][i]["Tweet text"]
    input_text = f'{text_column} : {tweet}\nLabel : '
    inputs = tokenizer(input_text, return_tensors="pt")

    with torch.no_grad():
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=10,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    response = generated[len(input_text):].strip()
    print(f"Sample {i:>2}: {response}")

In [ ]:
!nvidia-smi